In [1]:
# =============================================
# 02 - SATIŞ ANALİZİ
# SQL sorğuları işlədir → Excel-ə export edir
# =============================================

import sqlite3
import pandas as pd

# Bağlantı
conn = sqlite3.connect(r"C:\Users\Almaz\superstore_dashboard\superstore.db")

# 1. TOP 10 MƏHSUL
df1 = pd.read_sql("""
    SELECT product_name, ROUND(SUM(sales),2) AS umumi_satis,
           ROUND(SUM(profit),2) AS umumi_menfeet, COUNT(order_id) AS sifaris_sayi
    FROM orders GROUP BY product_name ORDER BY umumi_satis DESC LIMIT 10
""", conn)

# 2. İTKİ VERƏN MƏHSULLAR
df2 = pd.read_sql("""
    SELECT product_name, category, ROUND(SUM(profit),2) AS umumi_itki,
           COUNT(order_id) AS sifaris_sayi, ROUND(AVG(discount)*100,1) AS orta_endirim
    FROM orders WHERE profit < 0
    GROUP BY product_name ORDER BY umumi_itki ASC LIMIT 10
""", conn)

# 3. REGİON ANALİZİ
df3 = pd.read_sql("""
    SELECT region, ROUND(SUM(sales),2) AS umumi_satis,
           ROUND(SUM(profit),2) AS umumi_menfeet,
           ROUND(SUM(profit)/SUM(sales)*100,2) AS menfeet_marjasi,
           COUNT(DISTINCT customer_id) AS musteri_sayi
    FROM orders GROUP BY region ORDER BY umumi_satis DESC
""", conn)

# 4. ENDİRİM ANALİZİ
df4 = pd.read_sql("""
    SELECT CASE WHEN discount=0 THEN 'Endirimsiz'
                WHEN discount<=0.20 THEN '1-20%'
                WHEN discount<=0.40 THEN '21-40%'
                WHEN discount<=0.60 THEN '41-60%'
                ELSE '60%+' END AS endirim_qrupu,
           COUNT(order_id) AS sifaris_sayi,
           ROUND(SUM(profit),2) AS umumi_menfeet,
           ROUND(AVG(profit),2) AS orta_menfeet
    FROM orders GROUP BY endirim_qrupu ORDER BY orta_menfeet DESC
""", conn)

# 5. MÜŞTƏRİ ANALİZİ
df5 = pd.read_sql("""
    SELECT customer_name, segment, region,
           COUNT(DISTINCT order_id) AS sifaris_sayi,
           ROUND(SUM(sales),2) AS umumi_satis,
           ROUND(SUM(profit),2) AS umumi_menfeet,
           ROUND(AVG(discount)*100,1) AS orta_endirim
    FROM orders GROUP BY customer_name ORDER BY umumi_satis DESC LIMIT 10
""", conn)

conn.close()

# EXCEL-Ə EXPORT
with pd.ExcelWriter(r"C:\Users\Almaz\superstore_dashboard\dashboard\hesabat.xlsx", engine='openpyxl') as writer:
    df1.to_excel(writer, sheet_name='TOP10 Mehsul', index=False)
    df2.to_excel(writer, sheet_name='Itki Verenler', index=False)
    df3.to_excel(writer, sheet_name='Region Analizi', index=False)
    df4.to_excel(writer, sheet_name='Endirim Analizi', index=False)
    df5.to_excel(writer, sheet_name='Musteri Analizi', index=False)

print("✅ hesabat.xlsx yaradıldı!")
print("✅ hesabat.xlsx yaradıldı!")
print("✅ hesabat.xlsx yaradıldı!")

✅ hesabat.xlsx yaradıldı!
✅ hesabat.xlsx yaradıldı!
✅ hesabat.xlsx yaradıldı!


In [2]:
# =============================================
# SÜTUN ADLARINI DƏYIŞ + EXCEL-Ə EXPORT ET
# =============================================

df1.columns = ['Məhsul Adı', 'Ümumi Satış', 'Ümumi Mənfəət', 'Sifariş Sayı']
df2.columns = ['Məhsul Adı', 'Kateqoriya', 'Ümumi İtki', 'Sifariş Sayı', 'Orta Endirim %']
df3.columns = ['Region', 'Ümumi Satış', 'Ümumi Mənfəət', 'Mənfəət Marjası %', 'Müştəri Sayı']
df4.columns = ['Endirim Qrupu', 'Sifariş Sayı', 'Ümumi Mənfəət', 'Orta Mənfəət']
df5.columns = ['Müştəri Adı', 'Seqment', 'Region', 'Sifariş Sayı', 'Ümumi Satış', 'Ümumi Mənfəət', 'Orta Endirim %']

with pd.ExcelWriter(r"C:\Users\Almaz\superstore_dashboard\dashboard\hesabat.xlsx", engine='openpyxl') as writer:
    df1.to_excel(writer, sheet_name='TOP10 Məhsul', index=False)
    df2.to_excel(writer, sheet_name='İtki Verənlər', index=False)
    df3.to_excel(writer, sheet_name='Region Analizi', index=False)
    df4.to_excel(writer, sheet_name='Endirim Analizi', index=False)
    df5.to_excel(writer, sheet_name='Müştəri Analizi', index=False)

print("✅ hesabat.xlsx yaradıldı!")

✅ hesabat.xlsx yaradıldı!


In [3]:
# =============================================
# RFM MÜŞTƏRİ SEQMENTASİYASI
# =============================================

import sqlite3
import pandas as pd

conn = sqlite3.connect(r"C:\Users\Almaz\superstore_dashboard\superstore.db")
df = pd.read_sql("SELECT customer_id, customer_name, order_date, order_id, sales FROM orders", conn)
conn.close()

# Tarixi çevir
df['order_date'] = pd.to_datetime(df['order_date'])

# Referans tarix — datasetin ən son tarixi
ref_tarix = df['order_date'].max()

# RFM hesabla
rfm = df.groupby(['customer_id', 'customer_name']).agg(
    Recency  = ('order_date', lambda x: (ref_tarix - x.max()).days),
    Frequency= ('order_id',   'nunique'),
    Monetary = ('sales',      'sum')
).reset_index()

rfm['Monetary'] = rfm['Monetary'].round(2)
print(rfm.head(10))

  customer_id         customer_name  Recency  Frequency  Monetary
0    AA-10315            Alex Avila      184          5   5563.56
1    AA-10375          Allen Armold       19          9   1056.39
2    AA-10480          Andrew Allen      259          4   1790.51
3    AA-10645         Anna Andreadi       55          6   5086.93
4    AB-10015         Aaron Bergman      415          3    886.16
5    AB-10060       Adam Bellavance       54          8   7706.50
6    AB-10105         Adrian Barton       41         10  14473.57
7    AB-10150           Aimee Bixby       41          5    966.71
8    AB-10165           Alan Barnes       25          8   1113.84
9    AB-10255  Alejandro Ballentine      166          9    914.53


In [4]:
# =============================================
# RFM SEQMENT ETİKETLƏRİ
# =============================================

# Scoring — 1-dən 4-ə qədər qiymət ver
rfm['R_score'] = pd.qcut(rfm['Recency'],   q=4, labels=[4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1,2,3,4])
rfm['M_score'] = pd.qcut(rfm['Monetary'],  q=4, labels=[1,2,3,4])

rfm['RFM_score'] = rfm['R_score'].astype(int) + rfm['F_score'].astype(int) + rfm['M_score'].astype(int)

# Seqment ver
def segment(score):
    if score >= 10:
        return '⭐ VIP'
    elif score >= 8:
        return '💚 Loyal'
    elif score >= 6:
        return '🔵 Potensial'
    elif score >= 4:
        return '⚠️ Riskli'
    else:
        return '❌ İtmiş'

rfm['Seqment'] = rfm['RFM_score'].apply(segment)

print(rfm[['customer_name','Recency','Frequency','Monetary','RFM_score','Seqment']].head(15))
print("\nSeqment sayları:")
print(rfm['Seqment'].value_counts())

           customer_name  Recency  Frequency  Monetary  RFM_score      Seqment
0             Alex Avila      184          5   5563.56          6  🔵 Potensial
1           Allen Armold       19          9   1056.39          9      💚 Loyal
2           Andrew Allen      259          4   1790.51          4    ⚠️ Riskli
3          Anna Andreadi       55          6   5086.93          9      💚 Loyal
4          Aaron Bergman      415          3    886.16          3      ❌ İtmiş
5        Adam Bellavance       54          8   7706.50         10        ⭐ VIP
6          Adrian Barton       41         10  14473.57         11        ⭐ VIP
7            Aimee Bixby       41          5    966.71          5    ⚠️ Riskli
8            Alan Barnes       25          8   1113.84          8      💚 Loyal
9   Alejandro Ballentine      166          9    914.53          7  🔵 Potensial
10             Ann Blume       50          4   1515.86          6  🔵 Potensial
11         Alyssa Crouse      232          3    925.

In [6]:
# SÜTUN ADLARINI DƏYIŞ + RFM-i Excel-ə əlavə et
rfm_export = rfm[['customer_name','Recency','Frequency','Monetary','RFM_score','Seqment']].copy()
rfm_export.columns = ['Müştəri Adı', 'Son Alış (gün)', 'Sifariş Sayı', 'Ümumi Xərc', 'RFM Bal', 'Seqment']

with pd.ExcelWriter(r"C:\Users\Almaz\superstore_dashboard\dashboard\hesabat.xlsx", 
                    engine='openpyxl', mode='a') as writer:
    rfm_export.to_excel(writer, sheet_name='RFM Analizi', index=False)

print("✅ RFM vərəqi əlavə edildi!")



✅ RFM vərəqi əlavə edildi!


In [9]:
import plotly.express as px

seqment_df = rfm['Seqment'].value_counts().reset_index()
seqment_df.columns = ['Seqment', 'Müştəri Sayı']

fig = px.pie(
    seqment_df,
    values='Müştəri Sayı',
    names='Seqment',
    title='Müştəri Seqmentasiyası (RFM)',
    color_discrete_sequence=px.colors.qualitative.Set2,
    hole=0.4
)

fig.update_layout(
    title_font_size=20,
    legend_title='Seqment'
)

fig.write_html(r"C:\Users\Almaz\superstore_dashboard\dashboard\rfm_seqment.html")
print("✅ rfm_seqment.html yaradıldı!")

✅ rfm_seqment.html yaradıldı!


In [11]:
# =============================================
# PLOTLY - AYLIK SATIŞ TRENDİ
# =============================================

import sqlite3
import pandas as pd
import plotly.express as px

conn = sqlite3.connect(r"C:\Users\Almaz\superstore_dashboard\superstore.db")
df = pd.read_sql("SELECT order_date, sales, profit FROM orders", conn)
conn.close()

df['order_date'] = pd.to_datetime(df['order_date'])
df['ay'] = df['order_date'].dt.to_period('M').astype(str)

aylik = df.groupby('ay').agg(
    Satış  = ('sales', 'sum'),
    Mənfəət = ('profit', 'sum')
).reset_index()

fig = px.line(
    aylik,
    x='ay',
    y=['Satış', 'Mənfəət'],
    title='Aylıq Satış və Mənfəət Trendi',
    markers=True,
    color_discrete_map={'Satış': '#1f77b4', 'Mənfəət': '#2ca02c'}
)

fig.update_layout(
    title_font_size=20,
    xaxis_title='Ay',
    yaxis_title='Məbləğ ($)',
    legend_title='Göstərici',
    xaxis_tickangle=45
)

fig.write_html(r"C:\Users\Almaz\superstore_dashboard\dashboard\satis_trendi.html")
print("✅ satis_trendi.html yaradıldı!")

✅ satis_trendi.html yaradıldı!


In [ ]:
# =============================================
# PLOTLY - AYLIK SATIŞ TRENDİ
# =============================================

import sqlite3
import pandas as pd
import plotly.express as px

conn = sqlite3.connect(r"C:\Users\Almaz\superstore_dashboard\superstore.db")
df = pd.read_sql("SELECT order_date, sales, profit FROM orders", conn)
conn.close()

df['order_date'] = pd.to_datetime(df['order_date'])
df['ay'] = df['order_date'].dt.to_period('M').astype(str)

aylik = df.groupby('ay').agg(
    Satış  = ('sales', 'sum'),
    Mənfəət = ('profit', 'sum')
).reset_index()

fig = px.line(
    aylik,
    x='ay',
    y=['Satış', 'Mənfəət'],
    title='Aylıq Satış və Mənfəət Trendi',
    markers=True,
    color_discrete_map={'Satış': '#1f77b4', 'Mənfəət': '#2ca02c'}
)

fig.update_layout(
    title_font_size=20,
    xaxis_title='Ay',
    yaxis_title='Məbləğ ($)',
    legend_title='Göstərici',
    xaxis_tickangle=45
)

fig.write_html(r"C:\Users\Almaz\superstore_dashboard\dashboard\satis_trendi.html")
print("✅ satis_trendi.html yaradıldı!")

✅ satis_trendi.html yaradıldı!


In [12]:
# =============================================
# PLOTLY - ENDİRİM ANALİZİ
# =============================================

fig = px.bar(
    df4,
    x='Endirim Qrupu',
    y='Orta Mənfəət',
    title='Endirim Qruplarına Görə Orta Mənfəət',
    color='Orta Mənfəət',
    color_continuous_scale=['red', 'yellow', 'green'],
    text='Orta Mənfəət'
)

fig.update_traces(texttemplate='%{text:.1f}$', textposition='outside')
fig.update_layout(
    title_font_size=20,
    xaxis_title='Endirim Qrupu',
    yaxis_title='Orta Mənfəət ($)',
    coloraxis_showscale=False
)

fig.write_html(r"C:\Users\Almaz\superstore_dashboard\dashboard\endirim_analizi.html")
print("✅ endirim_analizi.html yaradıldı!")

✅ endirim_analizi.html yaradıldı!
